In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, '../Week-2-3-4')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,RobustScaler,TargetEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV


from sklearn.svm import SVR,NuSVR # regression
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# UHPC Modelling & Hyperparameter Tuning

## 1. Imports

In [14]:
raw = pd.read_csv("../Datasets/processed//uhpc_dataset/initial_cleaned.csv")
df_raw_20 = pd.read_csv("../Datasets/processed//uhpc_dataset/raw_dropped_features_20.csv")
df_raw_50 = pd.read_csv("../Datasets/processed//uhpc_dataset/raw_dropped_features_50.csv")
df_recoded_20 = pd.read_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features_20.csv")
df_recoded_50 = pd.read_csv("../Datasets/processed//uhpc_dataset/semantic_recoding_features_50.csv")

df_recoded_50 = df_recoded_50.drop(columns=['cement_grade']) # cement grade 38% missing, encodes sames info as cement type
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_recoded_50[fiber_cols] = df_recoded_50[fiber_cols].fillna(0)  # replacing with mean wrong

dfs = [raw,df_raw_20, df_raw_50, df_recoded_20, df_recoded_50]
df_names = ['raw','raw_20', 'raw_50', 'recoded_20', 'recoded_50']
models = [KNeighborsRegressor, NuSVR, SVR]
model_names_list = ['KNeighborsRegressor', 'NuSVR', 'SVR']


## 2. Load & Prepare Data

In [15]:
def prepare_data(df, target_col='cs_28d', train_size=0.70, random_state=42):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=1-train_size, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=random_state
    )
    
    return X_train, X_val, X_test, y_train, y_val, y_test


def identify_column_types(X):
    str_cols = X.select_dtypes(include='object').columns
    one_hot_columns = str_cols[X[str_cols].nunique() <= 10].tolist()
    k_fold_columns = str_cols[X[str_cols].nunique() > 10].tolist()
    numerical_cols = X.select_dtypes(include='number').columns.tolist()
    
    return numerical_cols, one_hot_columns, k_fold_columns


def create_preprocessor(X_train, numerical_cols, one_hot_columns, k_fold_columns, 
                        handle_unknown='ignore'):
    numerical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])
    
    #numerical_pipeline.fit(X_train[numerical_cols])
    
    preprocessor = ColumnTransformer([
        ('num', numerical_pipeline, numerical_cols),
        ('ohe', OneHotEncoder(handle_unknown=handle_unknown), one_hot_columns),
        ('target', TargetEncoder(cv=5), k_fold_columns)
    ])
    
    return preprocessor


def evaluate_model(y_true, y_pred, set_name=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r, _ = pearsonr(y_true, y_pred)
    
    metrics = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Correlation': r
    }
    
    if set_name:
        print(f"\n{set_name.upper()} SET PERFORMANCE")
        print("="*60)
        for metric_name, value in metrics.items():
            print(f"{metric_name}: {value:.4f}")
    
    return metrics

## 3. Utility Functions

**Key Pipeline Function:**
- `create_preprocessor()` creates ColumnTransformer with: numerical scaling + OneHotEncoder + TargetEncoder

In [16]:
for df, df_name in zip(dfs, df_names):
    for model_class, model_name in zip(models, model_names_list):
        
        target_col = 'cs_28d'
        X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df, target_col)
        
        X = df.drop(columns=[target_col])
        numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)
        
        preprocessor = create_preprocessor(X_train, numerical_cols, one_hot_columns, k_fold_columns)
        
        full_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('model', model_class())
        ])
        
        full_pipeline.fit(X_train, y_train)
        
        y_pred_train = full_pipeline.predict(X_train)
        y_pred_test = full_pipeline.predict(X_test)
        
        train_metrics = evaluate_model(y_train, y_pred_train)
        test_metrics = evaluate_model(y_test, y_pred_test)
        
        rmse_train = train_metrics['RMSE']
        rmse_test = test_metrics['RMSE']
        mae = test_metrics['MAE']
        r2 = test_metrics['R2']
        r = test_metrics['Correlation']
        
        print(f"Dataset: {df_name:12} | Model: {model_name:20} | RMSE test: {rmse_test:.4f} | RMSE train: {rmse_train:.4f} | MAE: {mae:.4f} | R2: {r2:.4f} | R: {r:.4f}")

Dataset: raw          | Model: KNeighborsRegressor  | RMSE test: 20.8437 | RMSE train: 18.8754 | MAE: 15.7714 | R2: 0.6837 | R: 0.8289
Dataset: raw          | Model: NuSVR                | RMSE test: 34.3521 | RMSE train: 34.0967 | MAE: 26.3789 | R2: 0.1409 | R: 0.5472
Dataset: raw          | Model: SVR                  | RMSE test: 34.1910 | RMSE train: 33.9257 | MAE: 26.0300 | R2: 0.1489 | R: 0.5477
Dataset: raw_20       | Model: KNeighborsRegressor  | RMSE test: 22.7774 | RMSE train: 20.7996 | MAE: 16.9305 | R2: 0.6223 | R: 0.7922
Dataset: raw_20       | Model: NuSVR                | RMSE test: 34.2614 | RMSE train: 34.0898 | MAE: 26.4358 | R2: 0.1454 | R: 0.5202
Dataset: raw_20       | Model: SVR                  | RMSE test: 34.0733 | RMSE train: 33.8226 | MAE: 26.0073 | R2: 0.1547 | R: 0.5343
Dataset: raw_50       | Model: KNeighborsRegressor  | RMSE test: 20.8388 | RMSE train: 18.8564 | MAE: 15.4231 | R2: 0.6838 | R: 0.8327
Dataset: raw_50       | Model: NuSVR                | R

## 4. Model Comparison Loop

**Pipeline Created:**
```python
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),  # Uses create_preprocessor()
    ('model', model_class())
])
```

In [17]:
def run_grid_search(pipeline, param_grid, X_train, X_val, X_test, y_train, y_val, y_test, model_label):
    total = 1
    for v in param_grid.values():
        total *= len(v)
    print(f"\nGridSearchCV will test {total} combinations for {model_label}...")

    gs = GridSearchCV(pipeline, param_grid, cv=5,
                      scoring='neg_mean_squared_error', n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)

    print("\n" + "="*80)
    print(f"BEST HYPERPARAMETERS — {model_label}")
    print("="*80)
    for param, value in gs.best_params_.items():
        print(f"  {param.replace('model__', '')}: {value}")
    print(f"Best CV RMSE: {np.sqrt(-gs.best_score_):.4f}")

    val_metrics  = evaluate_model(y_val,  gs.predict(X_val),  'Validation')
    test_metrics = evaluate_model(y_test, gs.predict(X_test), 'Test')

    summary = pd.DataFrame({
        'Metric': ['RMSE', 'MAE', 'R2', 'Correlation (R)'],
        'Validation': list(val_metrics.values()),
        'Test':       list(test_metrics.values())
    })
    print("\nRESULTS SUMMARY\n", summary.to_string(index=False))

    cv_df = pd.DataFrame(gs.cv_results_).sort_values('mean_test_score', ascending=False)
    param_cols = [c for c in cv_df.columns if c.startswith('param_model__')]
    top10 = cv_df[param_cols + ['mean_test_score']].head(10).copy()
    top10['CV_rmse'] = np.sqrt(-top10['mean_test_score'].values)
    top10.columns = [c.replace('param_model__', '') for c in top10.columns]
    print(f"\nTOP 10 COMBINATIONS\n{top10.to_string(index=False)}")

    return gs

## 5. Grid Search Helper Function

**Purpose:** Automates hyperparameter tuning for any pipeline

In [18]:
df = df_recoded_50
target_col = 'cs_28d'

X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df, target_col)

print(f"Train set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

X = df.drop(columns=[target_col])
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(X_train, numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore')

Train set size: 1450
Validation set size: 311
Test set size: 311


## 6. Hyperparameter Tuning Setup

**Preprocessing Pipeline Created:**
```python
preprocessor = create_preprocessor(...)  # ColumnTransformer
```
Used for all 3 models below (KNN, SVR, NuSVR)

### 6.1 KNN - Pipeline & Tuning

In [ ]:
# KNN PIPELINE CREATION
knn_pipeline = Pipeline([('preprocessor', preprocessor), ('model', KNeighborsRegressor())])

knn_grid = {
    'model__n_neighbors': list(range(1, 31)),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2, 3],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

gs_knn = run_grid_search(knn_pipeline, knn_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'KNN')


GridSearchCV will test 540 combinations for KNN...

BEST HYPERPARAMETERS — KNN
  metric: manhattan
  n_neighbors: 4
  p: 2
  weights: uniform
Best CV RMSE: 23.0466

VALIDATION SET PERFORMANCE
RMSE: 21.8017
MAE: 15.4430
R2: 0.5862
Correlation: 0.7889

TEST SET PERFORMANCE
RMSE: 19.5739
MAE: 14.4384
R2: 0.7211
Correlation: 0.8512

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   21.801693 19.573863
            MAE   15.442982 14.438423
             R2    0.586232  0.721058
Correlation (R)    0.788884  0.851240

TOP 10 COMBINATIONS
   metric  n_neighbors  p  weights  mean_test_score   CV_rmse
manhattan            4  2  uniform      -531.146903 23.046625
manhattan            3  1  uniform      -531.157126 23.046846
minkowski            8  1 distance      -534.786248 23.125446
manhattan            5  1  uniform      -534.875376 23.127373
manhattan            9  2 distance      -536.459088 23.161586
manhattan            7  2 distance      -536.728720 23.167406
manhat

### 6.2 SVR - Pipeline & Tuning

In [ ]:
# SVR PIPELINE CREATION
svr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', SVR(kernel='rbf'))])

svr_grid = {
    'model__C':        [128, 256, 512, 1024],
    'model__epsilon': [0.01, 0.1, 0.5, 1, 3]
}

gs_svr = run_grid_search(svr_pipeline, svr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'SVR')


GridSearchCV will test 20 combinations for SVR...

BEST HYPERPARAMETERS — SVR
  C: 1024
  epsilon: 3
Best CV RMSE: 26.4502

VALIDATION SET PERFORMANCE
RMSE: 24.9446
MAE: 18.8626
R2: 0.4583
Correlation: 0.6775

TEST SET PERFORMANCE
RMSE: 25.1504
MAE: 18.5195
R2: 0.5395
Correlation: 0.7457

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   24.944591 25.150383
            MAE   18.862645 18.519466
             R2    0.458336  0.539478
Correlation (R)    0.677476  0.745676

TOP 10 COMBINATIONS
   C  epsilon  mean_test_score   CV_rmse
1024     3.00      -699.612396 26.450187
1024     0.50      -700.348377 26.464096
1024     1.00      -702.800253 26.510380
1024     0.10      -705.364679 26.558703
1024     0.01      -708.573519 26.619044
 512     0.50      -715.369900 26.746400
 512     0.10      -716.768508 26.772533
 512     1.00      -718.038795 26.796246
 512     0.01      -718.074753 26.796917
 512     3.00      -720.527611 26.842645


### 6.3 NuSVR - Pipeline & Tuning

In [ ]:
# NuSVR PIPELINE CREATION
nusvr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', NuSVR(kernel='rbf'))])

nusvr_grid = {
    'model__C':   [128, 256, 512, 1024],
    'model__nu': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

gs_nusvr = run_grid_search(nusvr_pipeline, nusvr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'NuSVR')


GridSearchCV will test 36 combinations for NuSVR...

BEST HYPERPARAMETERS — NuSVR
  C: 1024
  nu: 0.6
Best CV RMSE: 26.2186

VALIDATION SET PERFORMANCE
RMSE: 25.0931
MAE: 19.1413
R2: 0.4519
Correlation: 0.6731

TEST SET PERFORMANCE
RMSE: 24.9437
MAE: 18.4779
R2: 0.5470
Correlation: 0.7480

RESULTS SUMMARY
          Metric  Validation      Test
           RMSE   25.093150 24.943658
            MAE   19.141322 18.477938
             R2    0.451865  0.547017
Correlation (R)    0.673123  0.747964

TOP 10 COMBINATIONS
   C  nu  mean_test_score   CV_rmse
1024 0.6      -687.412782 26.218558
1024 0.7      -692.700739 26.319209
1024 0.8      -693.510336 26.334584
1024 0.4      -695.100817 26.364765
1024 0.5      -695.552449 26.373328
1024 0.9      -699.599299 26.449939
1024 0.3      -699.979718 26.457130
 512 0.7      -714.415658 26.728555
 512 0.8      -715.157161 26.742422
 512 0.5      -715.186171 26.742965


## 7. Feature Analysis

**Preprocessing Pipeline:**
```python
preprocessor.fit(X_train, y_train)
X_transformed = preprocessor.transform(X_train)
```
Summary statistics of preprocessed features

In [22]:
target_col = 'cs_28d'
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(df_recoded_50, target_col)

X = df_recoded_50.drop(columns=[target_col])
numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)
preprocessor = create_preprocessor(X_train, numerical_cols, one_hot_columns, k_fold_columns)
preprocessor.fit(X_train, y_train)

# get column names
ohe_cols = preprocessor.named_transformers_['ohe'].get_feature_names_out(one_hot_columns).tolist()
all_cols = numerical_cols + ohe_cols + k_fold_columns

X_transformed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=all_cols
)

summary = pd.DataFrame({
    'mean':     X_transformed.mean().round(2),
    'median':   X_transformed.median().round(2),
    'std':      X_transformed.std().round(2),
    'min':      X_transformed.min().round(2),
    'max':      X_transformed.max().round(2),
    'cv (%)':   (X_transformed.std() / X_transformed.mean() * 100).round(1),
    'skewness': X_transformed.skew().round(2),
    'missing':  X_transformed.isna().sum()
})

print(summary.to_string())

                                    mean  median     std     min      max  cv (%)  skewness  missing
cement                             -0.02    0.00    0.89   -2.75     4.71 -5812.9      0.45        0
silica_fume                        -0.18    0.00    0.96   -2.06     4.46  -525.1      0.37        0
fly_ash                            59.95    0.00  144.51    0.00  1152.00   241.0      2.88        0
limestone_powder                   14.77    0.00   65.64    0.00  1058.20   444.3      6.65        0
quartz_powder                       0.96    0.00    1.80    0.00     8.27   187.8      1.70        0
glass_powder                       32.30    0.00  120.94    0.00  1067.00   374.5      4.72        0
rice_husk_ash                       2.47    0.00   24.02    0.00   481.06   973.1     12.95        0
metakaolin                          5.70    0.00   35.05    0.00   510.00   614.6      8.45        0
ggbfs                              40.69    0.00  142.35    0.00   750.00   349.8      3.93

## 8. Data Info & Missing Values

In [23]:
print(df_recoded_50.shape)
print(df_recoded_50.isnull().mean().sort_values(ascending=False).head(20))

(2072, 34)
sand_max_size       0.133205
curing_temp         0.075772
sp_amount           0.003861
cement              0.000000
cement_type         0.000000
silica_fume         0.000000
quartz_powder       0.000000
fly_ash             0.000000
fly_ash_type        0.000000
limestone_powder    0.000000
ggbfs               0.000000
slag                0.000000
slag_type           0.000000
nano_caco3          0.000000
nano_al2o3          0.000000
glass_powder        0.000000
rice_husk_ash       0.000000
metakaolin          0.000000
filler              0.000000
nano_sio2           0.000000
dtype: float64


## 9. Error Analysis by Category

Evaluates predictions from all 3 best models (KNN, SVR, NuSVR)

In [24]:
for gs, model_name in zip([gs_knn, gs_svr, gs_nusvr], ['KNN', 'SVR', 'NuSVR']):
    X_test_df = X_test.copy()
    X_test_df['y_true'] = y_test.values
    X_test_df['y_pred'] = gs.best_estimator_.predict(X_test)
    X_test_df['abs_error'] = abs(X_test_df['y_true'] - X_test_df['y_pred'])
    
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print('='*50)
    
    print("\n-- Cement Type --")
    print(X_test_df.groupby('cement_type')['abs_error'].mean().sort_values(ascending=False))
    
    print("\n-- Fiber vs No Fiber --")
    X_test_df['has_fiber'] = (X_test['fiber1_amount'] > 0).map({True: 'fiber', False: 'no_fiber'})
    print(X_test_df.groupby('has_fiber')['abs_error'].agg(['mean', 'count']))
    
    print("\n-- Curing Method --")
    print(X_test_df.groupby('curing_method')['abs_error'].mean().sort_values(ascending=False))


Model: KNN

-- Cement Type --
cement_type
Unknown            32.083333
HS_cement          18.471429
pozzolan_cement    18.385000
OPC_53             18.364265
OPC_42.5           14.746336
OPC_I              13.740027
OPC_52.5           13.553553
white_cement       11.540750
OPC_III             8.728000
CEM_II              6.379286
Name: abs_error, dtype: float64

-- Fiber vs No Fiber --
                mean  count
has_fiber                  
fiber      14.842062    214
no_fiber   13.547920     97

-- Curing Method --
curing_method
Water Curing        26.377500
Autoclave Curing    25.672143
Heat Curing         16.650488
Air Curing          16.596250
Steam Curing        15.341985
Standard Curing     12.757749
Hot Water Curing    12.464278
Name: abs_error, dtype: float64

Model: SVR

-- Cement Type --
cement_type
Unknown            39.723626
HS_cement          25.019903
pozzolan_cement    24.866519
OPC_53             24.086872
OPC_42.5           19.128573
OPC_I              18.505241
OPC_